# FloodVerify CV: final test on Nepal (run ONCE)
Runtime: **T4 GPU**. *Runtime > Run all*. About 30-40 min.

The final model is frozen: **M1 siamese U-Net, weighted CE, seed 0, `best.pt` (epoch 12), temperature T = 1.133**, all
chosen on Florence (validation). Nepal flooding was never used for training or for any choice. This notebook scores it
once and saves everything to `eval_extras/nepal_final/` on Drive. A guard stops a second run, so no number can be
"improved" by re-running. Nothing here may change the model, the temperature or any setting afterwards.

What it reports (all on Nepal, 2,476 crops):
1. Pixel level: localisation F1, per-class damage F1, xView2-style score, with 95% bootstrap intervals over crops.
2. Building level: found / missed, damage F1 per class, building-level score, with 95% intervals over images.
3. Calibration on predicted-building pixels with the Florence temperature (does it transfer?).
4. Evidence agreement on cells that contain buildings (the Gate 2 Nepal evidence vs label evidence).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # expect Tesla T4

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [2]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
RUN = "m1_siamese_unet_ce_s0"
OUTN = f"{D}/eval_extras/nepal_final"
import os
if os.path.exists(f"{OUTN}/DONE"):
    raise RuntimeError("STOP: Nepal has already been scored once. Read the saved results in " + OUTN + " instead of re-running.")
!mkdir -p "{OUTN}"
import datetime
open(f"{OUTN}/decision.txt", "w").write(
    f"Written {datetime.datetime.now().isoformat()} BEFORE any Nepal (test) result.\n"
    "Final model: m1_siamese_unet_ce_s0/best.pt (epoch 12), temperature 1.133, both chosen on hurricane-florence (val).\n"
    "Other models (all seeds of M1 CE, M1 Dice+focal, M0, M2) are scored on Nepal for comparison only,\n"
    "each with its best.pt chosen on Florence. No choice of model, checkpoint or temperature will change afterwards.\n")
print(open(f"{OUTN}/decision.txt").read())
!ls "{CK}/{RUN}"

Mounted at /content/drive
Written 2026-11-18T13:29:49.601997 BEFORE any Nepal (test) result.
Final model: m1_siamese_unet_ce_s0/best.pt (epoch 12), temperature 1.133, both chosen on hurricane-florence (val).
Other models (all seeds of M1 CE, M1 Dice+focal, M0, M2) are scored on Nepal for comparison only,
each with its best.pt chosen on Florence. No choice of model, checkpoint or temperature will change afterwards.

best.pt  config.yaml  eval_val.json  last.pt  metrics.jsonl  temperature.json


In [3]:
%cd /content
!rm -rf /content/FloodVerify
!git clone -q -b m3/eval-extras https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!git log --oneline -1
!pip install -q -e . --no-deps jsonschema
import sys; sys.path.insert(0, "/content/FloodVerify/src")

/content
/content/FloodVerify
4624338 (HEAD -> m3/eval-extras, origin/m3/eval-extras) cv/eval: paired bootstrap, building-pixel calibration, building-cell agreement, building-level evaluation and resolution ladder
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [4]:
# Faster data loading for prediction only (no effect on results: no augmentation, no randomness)
!sed -i 's/num_workers=2,/num_workers=8,/' /content/FloodVerify/src/floodsitrep/cv/eval/building_level.py /content/FloodVerify/src/floodsitrep/cv/eval/infer.py
!grep -n "num_workers=8" /content/FloodVerify/src/floodsitrep/cv/eval/building_level.py /content/FloodVerify/src/floodsitrep/cv/eval/infer.py

/content/FloodVerify/src/floodsitrep/cv/eval/building_level.py:176:    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=8,
/content/FloodVerify/src/floodsitrep/cv/eval/infer.py:26:def iter_predictions(model, xbd_root, manifest, role="val", batch_size=8, num_workers=8,


In [5]:
# Nepal crops are all in the official tier3 folder: copy only that, the manifest and the label evidence (about 5 min)
if not os.path.exists(f"{X}/processed/evidence_labels"):
    !mkdir -p /content/xbd/processed
    !cp -r "{D}/xbd/tier3" /content/xbd/
    !cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
    !unzip -q "{D}/xbd/processed.zip" -d /content/xbd/processed
import csv, collections
print(collections.Counter(r["split"] for r in csv.DictReader(open(MAN)) if r["event"] == "nepal-flooding"))  # expect tier3 only
!ls /content/xbd/tier3/images | grep -c nepal

Counter({'tier3': 2476})
1238


## Pass 1: pixel and building level (one prediction pass, about 10 min)

In [6]:
import json, numpy as np, pandas as pd
import floodsitrep.cv.eval.building_level as B
import floodsitrep.cv.eval.extras as EX
from floodsitrep.cv.eval import metrics as M
r = B.run(f"{CK}/{RUN}/best.pt", X, MAN, role="test", out_path=f"{OUTN}/building_level_test.npz")
print("crops", len(r["crop_ids"]), " images", len(r["img_ids"]), " buildings", int(r["bcms"].sum()),
      " skipped (no valid pixels)", int(r["skipped"].sum()))

crops 2476  images 619  buildings 41571  skipped (no valid pixels) 11


In [7]:
px_boot = M.bootstrap(r["crop_cms"], n_boot=1000)
b_point, b_boot = B.bootstrap_buildings(r["bcms"], r["comps"], r["falses"])
rows = []
for k_b, k_p in [("xview2_building_score", "xview2_pixel_score"), ("loc_f1", "loc_f1"), ("loc_recall", None),
                 ("loc_precision", None), ("dmg_f1_no_damage", "dmg_f1_no_damage"), ("dmg_f1_minor", "dmg_f1_minor"),
                 ("dmg_f1_major", "dmg_f1_major"), ("dmg_f1_destroyed", "dmg_f1_destroyed"), ("damage_f1_hm", "damage_f1_hm")]:
    row = {"metric": k_b.replace("xview2_building_score", "xview2-style score"),
           "building": b_boot[k_b]["value"], "bld 95% lo": b_boot[k_b]["lo"], "bld 95% hi": b_boot[k_b]["hi"]}
    if k_p:
        row.update({"pixel": px_boot[k_p]["value"], "px 95% lo": px_boot[k_p]["lo"], "px 95% hi": px_boot[k_p]["hi"]})
    rows.append(row)
NEPAL = pd.DataFrame(rows).round(4)
NEPAL.to_csv(f"{OUTN}/nepal_metrics.csv", index=False)
json.dump({"pixel_bootstrap_95": px_boot, "building_bootstrap_95": b_boot, "model": RUN,
           "checkpoint": "best.pt", "role": "test (nepal-flooding)"}, open(f"{OUTN}/nepal_metrics.json", "w"), indent=2)
print(NEPAL.to_string(index=False))

            metric  building  bld 95% lo  bld 95% hi  pixel  px 95% lo  px 95% hi
xview2-style score    0.4434      0.4221      0.4599 0.4151     0.3934     0.4349
            loc_f1    0.7579      0.7482      0.7665 0.7248     0.7192     0.7312
        loc_recall    0.7641      0.7483      0.7819    NaN        NaN        NaN
     loc_precision    0.7519      0.7368      0.7653    NaN        NaN        NaN
  dmg_f1_no_damage    0.5901      0.5498      0.6285 0.6222     0.5944     0.6478
      dmg_f1_minor    0.2533      0.2262      0.2774 0.2877     0.2641     0.3102
      dmg_f1_major    0.4622      0.4078      0.5170 0.5135     0.4712     0.5490
  dmg_f1_destroyed    0.1939      0.1553      0.2326 0.1401     0.1131     0.1702
      damage_f1_hm    0.3085      0.2778      0.3321 0.2824     0.2517     0.3106


In [8]:
# Where do Nepal's labelled buildings end up? (rows: true class; % of row)
bcm = r["bcms"].sum(0)
names = ["no damage", "minor", "major", "destroyed"]
df = pd.DataFrame(bcm, index=[f"true {n}" for n in names], columns=["missed"] + [f"pred {n}" for n in names])
print(df); print((df.div(df.sum(1), axis=0) * 100).round(1))
print("false building blobs:", int(r["falses"].sum()), "of", int(r["comps"].sum()))

                missed  pred no damage  pred minor  pred major  pred destroyed
true no damage    7648           13457        8880        1180              52
true minor        1040             817        2227         990              59
true major         800             117        1324        2085             394
true destroyed     319               4          22          48             108
                missed  pred no damage  pred minor  pred major  pred destroyed
true no damage    24.5            43.1        28.4         3.8             0.2
true minor        20.3            15.9        43.4        19.3             1.1
true major        16.9             2.5        28.1        44.2             8.3
true destroyed    63.7             0.8         4.4         9.6            21.6
false building blobs: 4207 of 16957


## Pass 2: calibration with the Florence temperature (about 10 min)

In [9]:
EX.collect(f"{CK}/{RUN}/best.pt", X, MAN, f"{OUTN}/calibration_sample_test.npz", role="test")
c = EX.load(f"{OUTN}/calibration_sample_test.npz")
T = json.load(open(f"{CK}/{RUN}/temperature.json"))["temperature"]
for t in (1.0, T):
    e, rows_ = EX.reliability(c["bld_logits"].astype(np.float64), c["bld_labels"], t=t)
    print(f"\nT = {t:.3f}   building-pixel ECE on Nepal {e:.4f}   ({len(c['bld_labels'])} pixels)")
    print(pd.DataFrame(rows_).round(3).to_string(index=False))


T = 1.000   building-pixel ECE on Nepal 0.1950   (1328230 pixels)
    bin  pixels  mean_confidence  accuracy
0.2-0.3    5158            0.284     0.160
0.3-0.4  130215            0.366     0.206
0.4-0.5  360148            0.454     0.274
0.5-0.6  363273            0.547     0.328
0.6-0.7  240945            0.646     0.403
0.7-0.8  133584            0.744     0.552
0.8-0.9   79657            0.845     0.752
0.9-1.0   15250            0.922     0.839

T = 1.133   building-pixel ECE on Nepal 0.1644   (1328230 pixels)
    bin  pixels  mean_confidence  accuracy
0.2-0.3    8484            0.283     0.162
0.3-0.4  182428            0.365     0.212
0.4-0.5  420652            0.453     0.286
0.5-0.6  358992            0.546     0.349
0.6-0.7  201127            0.644     0.456
0.7-0.8  100883            0.746     0.662
0.8-0.9   51267            0.840     0.794
0.9-1.0    4397            0.917     0.886


## Evidence agreement on Nepal (the Gate 2 evidence already on Drive; no model run)

In [10]:
from floodsitrep.cv.export_evidence import compare_with_labels
ag = EX.building_cell_agreement(f"{D}/evidence_predicted/{RUN}", f"{X}/processed/evidence_labels", prefix="nepal-flooding")
print(json.dumps(ag, indent=2))
json.dump(ag, open(f"{OUTN}/evidence_agreement_test.json", "w"), indent=2)

KeyboardInterrupt: 

## 5. Comparison models on Nepal (12 passes, about 10 min each; finished runs are skipped after a disconnect)
Each uses its own `best.pt`, chosen on Florence. Reported for comparison only.

In [11]:
FAMILIES = {"M1 weighted CE": "m1_siamese_unet_ce", "M1 Dice+focal": "m1_siamese_unet",
            "M0 Dice+focal": "m0_early_fusion_unet", "M2 SegFormer CE": "m2_siamese_segformer_ce"}
for fam, pre in FAMILIES.items():
    for s in range(3):
        run = f"{pre}_s{s}"; path = f"{OUTN}/cmp_{run}_test.npz"
        if run == RUN and not os.path.exists(path):
            import shutil; shutil.copy(f"{OUTN}/building_level_test.npz", path); continue   # final model: already scored
        if os.path.exists(path) or not os.path.exists(f"{CK}/{run}/best.pt"):
            continue
        print("scoring", run, flush=True)
        B.run(f"{CK}/{run}/best.pt", X, MAN, role="test", out_path=path)

scoring m1_siamese_unet_ce_s1
scoring m1_siamese_unet_ce_s2
scoring m1_siamese_unet_s0
scoring m1_siamese_unet_s1
scoring m1_siamese_unet_s2
scoring m0_early_fusion_unet_s0
scoring m0_early_fusion_unet_s1
scoring m0_early_fusion_unet_s2
scoring m2_siamese_segformer_ce_s0
scoring m2_siamese_segformer_ce_s1
scoring m2_siamese_segformer_ce_s2


In [12]:
rows = []
for fam, pre in FAMILIES.items():
    for s in range(3):
        path = f"{OUTN}/cmp_{pre}_s{s}_test.npz"
        if not os.path.exists(path): continue
        z = np.load(path)
        p = M.flat(M.summarise(z["crop_cms"].sum(0)))
        b = B.summarise_buildings(z["bcms"].sum(0), z["comps"].sum(), z["falses"].sum())
        rows.append({"model": fam, "seed": s, "pixel score": p["xview2_pixel_score"], "loc F1": p["loc_f1"],
                     "minor": p["dmg_f1_minor"], "major": p["dmg_f1_major"], "destroyed": p["dmg_f1_destroyed"],
                     "building score": b["xview2_building_score"], "bld recall": b["loc_recall"]})
CMP = pd.DataFrame(rows)
CMP.to_csv(f"{OUTN}/nepal_comparison.csv", index=False)
print(CMP.round(3).to_string(index=False)); print()
print(CMP.groupby("model", sort=False).agg(["mean", "std"]).drop(columns="seed").round(3).to_string())

          model  seed  pixel score  loc F1  minor  major  destroyed  building score  bld recall
 M1 weighted CE     0        0.415   0.725  0.288  0.513      0.140           0.443       0.764
 M1 weighted CE     1        0.394   0.731  0.285  0.487      0.115           0.432       0.771
 M1 weighted CE     2        0.440   0.742  0.300  0.541      0.157           0.465       0.807
  M1 Dice+focal     0        0.299   0.668  0.140  0.417      0.058           0.299       0.528
  M1 Dice+focal     1        0.324   0.686  0.226  0.499      0.064           0.309       0.560
  M1 Dice+focal     2        0.374   0.700  0.214  0.473      0.117           0.363       0.592
  M0 Dice+focal     0        0.355   0.672  0.207  0.266      0.131           0.349       0.546
  M0 Dice+focal     1        0.358   0.697  0.212  0.402      0.106           0.357       0.605
  M0 Dice+focal     2        0.293   0.616  0.177  0.205      0.075           0.285       0.439
M2 SegFormer CE     0        0.389   0.7

In [13]:
from google.colab import drive
drive.mount("/content/drive")
import pandas as pd, numpy as np
OUTN = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT/eval_extras/nepal_final"

# 1. Final model on Nepal: building and pixel metrics with 95% intervals
print(pd.read_csv(f"{OUTN}/nepal_metrics.csv").to_string(index=False))

# 2. Where Nepal's labelled buildings end up (counts, then % of each row)
z = np.load(f"{OUTN}/building_level_test.npz")
names = ["no damage", "minor", "major", "destroyed"]
df = pd.DataFrame(z["bcms"].sum(0), index=[f"true {n}" for n in names],
                  columns=["missed"] + [f"pred {n}" for n in names])
print(df)
print((df.div(df.sum(1), axis=0) * 100).round(1))
print("buildings:", int(z["bcms"].sum()), "| false building blobs:", int(z["falses"].sum()), "of", int(z["comps"].sum()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
            metric  building  bld 95% lo  bld 95% hi  pixel  px 95% lo  px 95% hi
xview2-style score    0.4434      0.4221      0.4599 0.4151     0.3934     0.4349
            loc_f1    0.7579      0.7482      0.7665 0.7248     0.7192     0.7312
        loc_recall    0.7641      0.7483      0.7819    NaN        NaN        NaN
     loc_precision    0.7519      0.7368      0.7653    NaN        NaN        NaN
  dmg_f1_no_damage    0.5901      0.5498      0.6285 0.6222     0.5944     0.6478
      dmg_f1_minor    0.2533      0.2262      0.2774 0.2877     0.2641     0.3102
      dmg_f1_major    0.4622      0.4078      0.5170 0.5135     0.4712     0.5490
  dmg_f1_destroyed    0.1939      0.1553      0.2326 0.1401     0.1131     0.1702
      damage_f1_hm    0.3085      0.2778      0.3321 0.2824     0.2517     0.3106
                missed  pred no damage  pred minor 

## Mark as done (this is what prevents a second run), then stop billing

In [ ]:
open(f"{OUTN}/DONE", "w").write("Nepal scored once with " + RUN + "/best.pt, T from Florence.\n")
!ls -la "{OUTN}"
from google.colab import runtime
runtime.unassign()